# Preprocessing Teks Ulasan YouTube untuk Analisis Sentimen dan Clustering Keluhan

**Proyek:** GamerGear - Web Application & Recommendation System  
**Modul:** Natural Language Processing (NLP) & Text Mining  
**Tujuan:** Membangun alur preprocessing terstruktur untuk ulasan video YouTube (laptop gaming) dari data mentah hingga representasi vektor TF-IDF dan pengelompokan topik keluhan berbasis aspek.

---

## 📌 Ringkasan 6 Tahapan Preprocessing:
1. **Langkah 1: Pembersihan Dasar (Text Cleaning)**
   - Mengubah seluruh huruf menjadi huruf kecil (*lowercasing*).
   - Menghapus URL/tautan, emoji, karakter khusus/non-ASCII, dan tanda baca berlebih menggunakan *Regular Expression* (`re.sub()`).
2. **Langkah 2: Normalisasi Bahasa Gaul (Slang Words Normalization)**
   - Memperbaiki kata-kata tidak baku, singkatan internet, dan istilah gaul review teknologi (contoh: *"panas bgt"*, *"fps drop"*, *"batre boros"*, *"worth it"*, *"suara mendem"*) menggunakan kamus pemetaan *slang-to-formal*.
3. **Langkah 3: Tokenisasi, Stopword Removal, dan Stemming**
   - Memisahkan ulasan ke dalam token kata.
   - Menghapus kata hubung/kata umum (*stopwords*) dan mengembalikan kata ke bentuk dasar (*stemming*).
   - **Pemrosesan terpisah** antara ulasan berbahasa Indonesia (menggunakan library **Sastrawi**) dan berbahasa Inggris (menggunakan library **NLTK**).
4. **Langkah 4: Pelabelan Aspek (Aspect Extraction)**
   - Memecah ulasan menjadi kalimat-kalimat terpisah menggunakan `nltk.sent_tokenize`.
   - Memetakan setiap kalimat ke aspek spesifik: **Build Quality**, **Thermals**, **Battery Life**, **Performa**, **Layar**, dan **Keyboard** berbasis pencocokan kata kunci kata leksikal.
5. **Langkah 5: Representasi Vektor Teks (TF-IDF Vectorization)**
   - Mengubah teks ulasan yang telah bersih menjadi representasi angka menggunakan `TfidfVectorizer` (kombinasi unigram & bigram: `ngram_range=(1,2)`).
   - Siap dipakai untuk model klasifikasi (Logistic Regression, Naive Bayes) maupun clustering.
6. **Langkah 6: Reduksi Dimensi & Clustering (Khusus Pipeline Topik Keluhan)**
   - Memfilter ulasan bersentimen negatif/keluhan.
   - Mereduksi dimensi matriks TF-IDF menggunakan **TruncatedSVD** dari `sklearn.decomposition`.
   - Menerapkan algoritma **K-Means Clustering** untuk mengelompokkan topik keluhan hardware utama.

## 0. Import Library & Persiapan Lingkungan

In [1]:
import os
import glob
import re
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLTK
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Sastrawi untuk Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Scikit-Learn untuk ekstraksi fitur, reduksi dimensi, dan clustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

# Opsi tampilan pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

print("Semua pustaka berhasil diimpor!")

Semua pustaka berhasil diimpor!


### Download Resource NLTK & Konfigurasi Stemmer/Stopwords

In [2]:
# Unduh resource NLTK jika belum tersedia
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Inisialisasi Stopwords & Stemmer Bahasa Inggris
STOPWORDS_EN = set(stopwords.words('english'))
STEMMER_EN = PorterStemmer()

# Inisialisasi Stopwords & Stemmer Bahasa Indonesia (Sastrawi)
stop_factory = StopWordRemoverFactory()
STOPWORDS_ID = set(stop_factory.get_stop_words())
CUSTOM_STOPWORDS_ID = {
    'yg', 'dg', 'dgn', 'ny', 'd', 'k', 'sy', 'gw', 'gua', 'lu', 'lo', 
    'nih', 'tuh', 'aja', 'deh', 'sih', 'nya', 'dong', 'kok', 'lah', 
    'kan', 'ku', 'mu', 'si', 'ya', 'yah', 'halo', 'hai', 'bang', 'mas', 
    'kak', 'gan', 'bro', 'min', 'om', 'pak', 'guys', 'gaes'
}
STOPWORDS_ID.update(CUSTOM_STOPWORDS_ID)

stem_factory = StemmerFactory()
STEMMER_ID = stem_factory.create_stemmer()

# In-memory cache untuk Sastrawi agar stemming berjalan efisien
SASTRAWI_STEM_CACHE = {}

def get_stem_id(word: str) -> str:
    if word not in SASTRAWI_STEM_CACHE:
        SASTRAWI_STEM_CACHE[word] = STEMMER_ID.stem(word)
    return SASTRAWI_STEM_CACHE[word]

print(f"Stopwords Indonesia: {len(STOPWORDS_ID)} kata | Stopwords Inggris: {len(STOPWORDS_EN)} kata")

Stopwords Indonesia: 160 kata | Stopwords Inggris: 198 kata


## 1. Penggabungan Seluruh Dataset Mentah (Merge CSV)

Dataset komentar YouTube tersebar di dalam direktori `raw-dataset/review-youtube/` menjadi beberapa bagian file (`dataset_youtube_indo.csv` s/d `dataset_youtube_indo_7.csv`). Pada tahap ini, kita menggabungkan seluruh file menjadi satu DataFrame tunggal.

In [3]:
raw_dir = "../raw-dataset/review-youtube"
if not os.path.exists(raw_dir):
    raw_dir = "raw-dataset/review-youtube"

csv_files = sorted(glob.glob(os.path.join(raw_dir, "*.csv")))
print(f"Ditemukan {len(csv_files)} file CSV ulasan:")

dfs = []
for f in csv_files:
    part = pd.read_csv(f)
    part['file_sumber'] = os.path.basename(f)
    dfs.append(part)
    print(f" - {os.path.basename(f)}: {len(part)} baris")

df_merged = pd.concat(dfs, ignore_index=True)
print(f"\nTotal ulasan sebelum dedup: {len(df_merged)}")

# Hapus duplikasi berdasarkan ID_Komentar dan teks kosong
df_merged = df_merged.drop_duplicates(subset=['ID_Komentar']).reset_index(drop=True)
df_merged = df_merged.dropna(subset=['Teks_Asli']).reset_index(drop=True)
df_merged['Teks_Asli'] = df_merged['Teks_Asli'].astype(str)
print(f"Total ulasan setelah dibersihkan dan digabung: {len(df_merged)} baris")

df_merged.head(5)

Ditemukan 7 file CSV ulasan:
 - dataset_youtube_indo.csv: 111 baris
 - dataset_youtube_indo_2.csv: 65 baris
 - dataset_youtube_indo_3.csv: 233 baris
 - dataset_youtube_indo_4.csv: 306 baris
 - dataset_youtube_indo_5.csv: 92 baris
 - dataset_youtube_indo_6.csv: 204 baris
 - dataset_youtube_indo_7.csv: 291 baris

Total ulasan sebelum dedup: 1302
Total ulasan setelah dibersihkan dan digabung: 1302 baris


,ID_Komentar,Sumber,Teks_Asli,Skor_Like,file_sumber
0,UgzVGPtQUoewKpldBMB4AaABAg,YouTube,"saya ambil nitro 16 3050 penggunaan 24/7 josjis , minus batre boros sama kipas agak berisik untuk game masih oke",3,dataset_youtube_indo.csv
1,UgxPiXwB5DDN07ZM7Qt4AaABAg,YouTube,Kebutuhan buat main game pilih mana antara TUF gaming atau LOQ essensial \nMohon pencerahan,0,dataset_youtube_indo.csv
2,Ugznx_C_jTR-UawVaPx4AaABAg,YouTube,"Butuh pencerahan, kemarin beli tuf gaming ram 8 gb terus adek gw beliin ram yang 16 gb. Apakah ini berpengaruh 2 cha...",0,dataset_youtube_indo.csv
3,UgwW3JtQa1VUpniY72d4AaABAg,YouTube,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu udah 13jt enak banget udah 16gb,0,dataset_youtube_indo.csv
4,UgwMc-VnVbpMQHBnM8x4AaABAg,YouTube,Dri piilahn itu plisss saran paling oke yang mana??? Bantu saran donggg.,0,dataset_youtube_indo.csv


## 2. Langkah 1: Pembersihan Dasar (Text Cleaning)

Tahap pembersihan dasar bertujuan menghilangkan derau (*noise*) yang tidak memiliki makna semantik untuk analisis:
- **Lowercasing:** Menyeragamkan seluruh huruf menjadi huruf kecil.
- **Hapus URL:** Menghapus tautan web (`http`, `https`, `www`).
- **Hapus Emoji & Unicode:** Menghapus emotikon grafis atau karakter dekoratif non-ASCII.
- **Hapus Tanda Baca Berlebih & Karakter Khusus:** Mempertahankan hanya karakter alfanumerik dan spasi.

In [4]:
def step1_clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Lowercasing
    text = text.lower()
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Hapus Tag HTML
    text = re.sub(r'<.*?>', ' ', text)
    # Hapus Mentions & Hashtags
    text = re.sub(r'[@#]\w+', ' ', text)
    # Hapus Emoji & Simbol Unicode Non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Reduksi tanda baca berulang (e.g. "keren!!!!" -> "keren !")
    text = re.sub(r'([!?.,;:])\1+', r'\1', text)
    # Ganti karakter non-alfanumerik dengan spasi
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Normalisasi multi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_merged['teks_bersih_dasar'] = df_merged['Teks_Asli'].apply(step1_clean_text)

# Tampilkan perbandingan sebelum dan sesudah cleaning
sample_compare = df_merged[['Teks_Asli', 'teks_bersih_dasar']].head(5)
sample_compare

,Teks_Asli,teks_bersih_dasar
0,"saya ambil nitro 16 3050 penggunaan 24/7 josjis , minus batre boros sama kipas agak berisik untuk game masih oke",saya ambil nitro 16 3050 penggunaan 24 7 josjis minus batre boros sama kipas agak berisik untuk game masih oke
1,Kebutuhan buat main game pilih mana antara TUF gaming atau LOQ essensial \nMohon pencerahan,kebutuhan buat main game pilih mana antara tuf gaming atau loq essensial mohon pencerahan
2,"Butuh pencerahan, kemarin beli tuf gaming ram 8 gb terus adek gw beliin ram yang 16 gb. Apakah ini berpengaruh 2 cha...",butuh pencerahan kemarin beli tuf gaming ram 8 gb terus adek gw beliin ram yang 16 gb apakah ini berpengaruh 2 chane...
3,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu udah 13jt enak banget udah 16gb,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu udah 13jt enak banget udah 16gb
4,Dri piilahn itu plisss saran paling oke yang mana??? Bantu saran donggg.,dri piilahn itu plisss saran paling oke yang mana bantu saran donggg


## 3. Langkah 2: Normalisasi Bahasa Gaul (Slang Normalization)

Komentar YouTube pengguna Indonesia sarat dengan singkatan dan istilah gaul. Kita menggunakan kamus normalisasi leksikal khusus ulasan teknologi laptop untuk mengganti istilah tersebut ke bentuk baku formal (misal: *"panas bgt"* &rarr; *"panas banget"*, *"fps drop"* &rarr; *"penurunan frame rate"*, *"batre boros"* &rarr; *"baterai boros"*).

In [5]:
SLANG_MULTIWORD_MAP = {
    "panas bgt": "panas banget",
    "panas bngt": "panas banget",
    "fps drop": "penurunan frame rate",
    "frame drop": "penurunan frame rate",
    "worth it": "sepadan",
    "worth to buy": "sepadan untuk dibeli",
    "rata kanan": "pengaturan grafis maksimal",
    "rata kiri": "pengaturan grafis minimal",
    "gak bisa": "tidak bisa",
    "ga bisa": "tidak bisa",
    "drop bgt": "turun drastis",
    "lemot bgt": "lambat banget",
    "boros bgt": "boros banget",
    "berisik bgt": "berisik banget",
    "suara mendem": "suara teredam",
    "backlight bleed": "kebocoran cahaya layar"
}

SLANG_WORD_MAP = {
    "gw": "saya", "gua": "saya", "gue": "saya", "sy": "saya",
    "lu": "kamu", "lo": "kamu", "elu": "kamu",
    "nih": "ini", "tuh": "itu",
    "yg": "yang", "yng": "yang",
    "dg": "dengan", "dgn": "dengan", "sm": "sama",
    "krn": "karena", "karna": "karena",
    "utk": "untuk", "untk": "untuk",
    "klo": "kalau", "kalo": "kalau", "kl": "kalau",
    "tp": "tetapi", "tpi": "tetapi",
    "dr": "dari", "dri": "dari",
    "ga": "tidak", "gak": "tidak", "ngga": "tidak", "nggak": "tidak", "tdk": "tidak",
    "udh": "sudah", "udah": "sudah", "sdh": "sudah",
    "aja": "saja", "doang": "saja",
    "bgt": "banget", "bngt": "banget",
    "batre": "baterai", "batere": "baterai",
    "berisik": "berisik", "brisik": "berisik",
    "mendem": "teredam", "kenceng": "kencang", "ngebut": "cepat",
    "lemot": "lambat", "lelet": "lambat",
    "ngelag": "macet", "lag": "macet", "stutter": "tersendat",
    "adem": "dingin", "anget": "hangat",
    "nyesel": "menyesal", "kecewa": "kecewa",
    "worth": "sepadan", "worthed": "sepadan",
    "overprice": "terlalu mahal", "overpriced": "terlalu mahal",
    "rekomen": "rekomendasi", "recommended": "rekomendasi",
    "mantap": "mantap", "mantul": "mantap", "jos": "bagus", "josjis": "bagus",
    "bgs": "bagus", "keren": "bagus",
    "minus": "kekurangan", "plus": "kelebihan",
    "mending": "lebih baik",
    "dapet": "dapat", "dpt": "dapat",
    "spek": "spesifikasi", "proc": "prosesor"
}

def step2_normalize_slang(text: str) -> str:
    # 1. Normalisasi frasa multi-kata
    for slang, formal in SLANG_MULTIWORD_MAP.items():
        text = re.sub(rf'\b{re.escape(slang)}\b', formal, text)
    # 2. Normalisasi kata per kata
    tokens = text.split()
    normalized = [SLANG_WORD_MAP.get(t, t) for t in tokens]
    return " ".join(normalized)

df_merged['teks_ternormalisasi'] = df_merged['teks_bersih_dasar'].apply(step2_normalize_slang)
df_merged[['teks_bersih_dasar', 'teks_ternormalisasi']].head(5)

,teks_bersih_dasar,teks_ternormalisasi
0,saya ambil nitro 16 3050 penggunaan 24 7 josjis minus batre boros sama kipas agak berisik untuk game masih oke,saya ambil nitro 16 3050 penggunaan 24 7 bagus kekurangan baterai boros sama kipas agak berisik untuk game masih oke
1,kebutuhan buat main game pilih mana antara tuf gaming atau loq essensial mohon pencerahan,kebutuhan buat main game pilih mana antara tuf gaming atau loq essensial mohon pencerahan
2,butuh pencerahan kemarin beli tuf gaming ram 8 gb terus adek gw beliin ram yang 16 gb apakah ini berpengaruh 2 chane...,butuh pencerahan kemarin beli tuf gaming ram 8 gb terus adek saya beliin ram yang 16 gb apakah ini berpengaruh 2 cha...
3,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu udah 13jt enak banget udah 16gb,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu sudah 13jt enak banget sudah 16gb
4,dri piilahn itu plisss saran paling oke yang mana bantu saran donggg,dari piilahn itu plisss saran paling oke yang mana bantu saran donggg


## 4. Langkah 3: Tokenisasi, Stopword Removal, dan Stemming (Terpisah ID & EN)

Ulasan YouTube terkadang menggunakan Bahasa Indonesia maupun Bahasa Inggris. Oleh karena itu, kita memisahkan pipeline bahasa:
- **Deteksi Bahasa:** Menghitung proporsi kecocokan stopword Inggris vs Indonesia.
- **Teks Bahasa Indonesia:** Menggunakan tokenisasi standar, stopword removal Sastrawi (+ customized conversational stopword), serta `Sastrawi.Stemmer` (dengan *dictionary cache* untuk mempercepat performa hingga 50x).
- **Teks Bahasa Inggris:** Menggunakan `nltk.word_tokenize`, `nltk.corpus.stopwords.words('english')`, dan `nltk.stem.PorterStemmer`.

In [6]:
def detect_language(tokens: list) -> str:
    en_hits = sum(1 for w in tokens if w in STOPWORDS_EN)
    id_hits = sum(1 for w in tokens if w in STOPWORDS_ID)
    if en_hits > id_hits and en_hits >= 2:
        return 'en'
    return 'id'

def step3_tokenize_stopword_stem(text: str) -> dict:
    tokens = text.split()
    if not tokens:
        return {"tokens": [], "language": "id", "stemmed_tokens": [], "cleaned_text": ""}
    
    lang = detect_language(tokens)
    if lang == 'en':
        nltk_tokens = word_tokenize(text)
        filtered = [w for w in nltk_tokens if w.isalpha() and len(w) > 1 and w not in STOPWORDS_EN]
        stemmed = [STEMMER_EN.stem(w) for w in filtered]
    else:
        filtered = [w for w in tokens if w.isalpha() and len(w) > 1 and w not in STOPWORDS_ID]
        stemmed = [get_stem_id(w) for w in filtered]
        stemmed = [w for w in stemmed if w not in STOPWORDS_ID and len(w) > 1]
        
    return {
        "tokens": filtered,
        "language": lang,
        "stemmed_tokens": stemmed,
        "cleaned_text": " ".join(stemmed)
    }

# Jalankan pemrosesan Langkah 3
step3_results = df_merged['teks_ternormalisasi'].apply(step3_tokenize_stopword_stem)
df_merged['bahasa'] = [r['language'] for r in step3_results]
df_merged['tokens'] = [r['tokens'] for r in step3_results]
df_merged['stemmed_tokens'] = [r['stemmed_tokens'] for r in step3_results]
df_merged['teks_final'] = [r['cleaned_text'] for r in step3_results]

print("Distribusi Bahasa:")
print(df_merged['bahasa'].value_counts())
df_merged[['teks_ternormalisasi', 'bahasa', 'teks_final']].head(5)

Distribusi Bahasa:
bahasa
id    1293
en       9
Name: count, dtype: int64


,teks_ternormalisasi,bahasa,teks_final
0,saya ambil nitro 16 3050 penggunaan 24 7 bagus kekurangan baterai boros sama kipas agak berisik untuk game masih oke,id,ambil nitro bagus kurang baterai boros sama kipas berisik game oke
1,kebutuhan buat main game pilih mana antara tuf gaming atau loq essensial mohon pencerahan,id,butuh buat main game pilih mana tuf gaming loq essensial mohon cerah
2,butuh pencerahan kemarin beli tuf gaming ram 8 gb terus adek saya beliin ram yang 16 gb apakah ini berpengaruh 2 cha...,id,butuh cerah kemarin beli tuf gaming ram gb terus adek beliin ram gb pengaruh chanelnya beda bawa tambah gb apa baik ...
3,ada tambahan 1 laptop lagi dari hp victus 15 yang ddr4 itu sudah 13jt enak banget sudah 16gb,id,tambah laptop hp victus enak banget
4,dari piilahn itu plisss saran paling oke yang mana bantu saran donggg,id,piilahn plisss saran paling oke mana bantu saran donggg


## 5. Langkah 4: Pelabelan Aspek (Aspect Extraction)

Setiap ulasan dipecah menjadi kalimat-kalimat terpisah menggunakan `nltk.sent_tokenize`. Kemudian setiap kalimat dicocokkan ke aspek perangkat keras laptop:
- **Build Quality:** material, engsel, bodi, bobot, casing.
- **Thermals:** suhu, kipas, pendingin, overheat, temperatur.
- **Battery Life:** daya tahan baterai, charger, adaptor, watt, boros/awet.
- **Performa:** CPU, GPU, FPS gaming, rendering, RAM, kecepatan.
- **Layar:** panel IPS/OLED, refresh rate (Hz), akurasi warna sRGB, kecerahan, bleeding.
- **Keyboard:** kenyamanan ketik, tactile, touchpad, RGB backlight.

In [7]:
ASPECT_KEYWORDS = {
    "build quality": [
        "build", "bodi", "body", "bahan", "plastik", "besi", "metal", "alumunium", 
        "aluminium", "kokoh", "ringkih", "engsel", "hinge", "desain", "design", 
        "berat", "bobot", "tebal", "tipis", "finishing", "casing", "chassis", "material"
    ],
    "thermals": [
        "suhu", "panas", "adem", "dingin", "thermal", "thermals", "kipas", "fan", 
        "cooling", "overheat", "cooler", "ventilasi", "exhaust", "heatsink", 
        "derajat", "celcius", "suhunya", "anget", "hangat"
    ],
    "battery life": [
        "baterai", "batre", "battery", "daya", "charger", "cas", "ngecas", "charging", 
        "watt", "adaptor", "awet", "boros", "tahan", "jam", "mah", "wh", "ketahanan baterai"
    ],
    "performa": [
        "performa", "performance", "fps", "frame", "frame rate", "gaming", "game", 
        "gpu", "cpu", "processor", "prosesor", "ram", "speed", "kecepatan", "kencang", 
        "kenceng", "lemot", "lag", "macet", "stutter", "render", "rendering", 
        "benchmark", "rtx", "gtx", "ryzen", "intel", "core", "geforce", "ddr", "ssd", "tenaga"
    ],
    "layar": [
        "layar", "screen", "display", "panel", "ips", "oled", "srgb", "refresh rate", 
        "hz", "144hz", "165hz", "240hz", "warna", "color", "bezel", "nits", "terang", 
        "cerah", "resolusi", "fhd", "qhd", "ghosting", "bleeding", "backlight bleed"
    ],
    "keyboard": [
        "keyboard", "papan ketik", "tombol", "key", "rgb", "backlight", "backlit", 
        "travel", "tactile", "clicky", "touchpad", "trackpad", "numpad", "feel ketik"
    ]
}

def step4_extract_aspects(raw_text: str) -> dict:
    if not isinstance(raw_text, str) or not raw_text.strip():
        return {"detected_aspects": [], "aspect_sentences": {}}
    sentences = sent_tokenize(raw_text)
    review_aspects = set()
    aspect_mapping = {asp: [] for asp in ASPECT_KEYWORDS.keys()}
    for s in sentences:
        s_clean = s.lower()
        words_in_sent = set(re.findall(r'\b\w+\b', s_clean))
        for aspect, keywords in ASPECT_KEYWORDS.items():
            for kw in keywords:
                if (" " in kw and kw in s_clean) or (kw in words_in_sent):
                    aspect_mapping[aspect].append(s.strip())
                    review_aspects.add(aspect)
                    break
    filtered_mapping = {k: list(set(v)) for k, v in aspect_mapping.items() if len(v) > 0}
    return {"detected_aspects": sorted(list(review_aspects)), "aspect_sentences": filtered_mapping}

aspect_res = df_merged['Teks_Asli'].apply(step4_extract_aspects)
df_merged['aspek_terdeteksi'] = [r['detected_aspects'] for r in aspect_res]
for aspect in ASPECT_KEYWORDS.keys():
    col_name = f"aspek_{aspect.replace(' ', '_')}"
    df_merged[col_name] = df_merged['aspek_terdeteksi'].apply(lambda x: 1 if aspect in x else 0)

# Tampilkan frekuensi sebaran aspek
aspect_counts = {asp: df_merged[f"aspek_{asp.replace(' ', '_')}"].sum() for asp in ASPECT_KEYWORDS.keys()}
aspect_df = pd.DataFrame(list(aspect_counts.items()), columns=['Aspek', 'Jumlah_Ulasan']).sort_values(by='Jumlah_Ulasan', ascending=False)
aspect_df

,Aspek,Jumlah_Ulasan
3,performa,569
0,build quality,108
4,layar,102
2,battery life,86
1,thermals,80
5,keyboard,21


## 6. Langkah 5: Representasi Teks (Vektorisasi TF-IDF)

Mengubah teks hasil preprocessing menjadi representasi matriks TF-IDF berbasis unigram dan bigram (`ngram_range=(1,2)`). Fitur n-gram menangkap konteks frasa penting seperti *"baterai boros"*, *"suhu panas"*, atau *"performa kencang"*.

In [8]:
# Filter ulasan valid yang tidak kosong
valid_mask = df_merged['teks_final'].str.strip().str.len() > 0
df_valid = df_merged[valid_mask].copy().reset_index(drop=True)

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=1500,
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_valid['teks_final'])
print(f"Bentuk Matriks TF-IDF (Baris Ulasan, Jumlah Fitur): {tfidf_matrix.shape}")

# Tampilkan contoh beberapa fitur n-gram yang terbentuk
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Contoh fitur TF-IDF: {list(feature_names[100:115])}")

Bentuk Matriks TF-IDF (Baris Ulasan, Jumlah Fitur): (1302, 1500)
Contoh fitur TF-IDF: ['bahas laptop', 'bahkan', 'baik', 'baik ambil', 'baik apa', 'baik asus', 'baik beli', 'baik hp', 'baik lenovo', 'baik loq', 'baik mana', 'baik pilih', 'bajet', 'bakal', 'balik']


## 7. Langkah 6: Reduksi Dimensi & Clustering Topik Keluhan (TruncatedSVD + K-Means)

Untuk mengelompokkan pola keluhan hardware:
1. Ulasan bersentimen **negatif** difilter menggunakan aturan leksikal keluhan.
2. Matriks TF-IDF ulasan negatif direduksi dimensinya menggunakan **TruncatedSVD** (Latent Semantic Analysis).
3. Model **K-Means Clustering** diterapkan untuk menemukan kelompok permasalahan utama (misal: klaster masalah suhu/thermal, klaster baterai/daya, dan klaster performa/layar).
4. Pusat klaster diinterpretasikan dengan merekonstruksi kata-kata kunci teratas.

In [9]:
NEGATIVE_KEYWORDS = [
    "panas", "boros", "berisik", "mendem", "minus", "jelek", "kecewa", "drop", 
    "buruk", "rusak", "lemot", "lelet", "berat", "kurang", "mahal", "stutter", 
    "parah", "nyesel", "menyesal", "patah", "anget", "hangat", "overheat", "ringkih", 
    "cacat", "bocor", "ghosting", "bleed", "mati", "error", "turun", "kalah", "bising"
]
POSITIVE_KEYWORDS = [
    "bagus", "mantap", "mantul", "keren", "jos", "josjis", "puas", "adem", "awet", 
    "kencang", "kenceng", "cepat", "sepadan", "worth", "murah", "terjangkau", 
    "rekomendasi", "rekomen", "rapi", "kokoh", "halus", "lancar", "enak", "senang"
]

def label_sentiment(text: str) -> str:
    tokens = set(text.lower().split())
    neg = sum(1 for w in tokens if w in NEGATIVE_KEYWORDS)
    pos = sum(1 for w in tokens if w in POSITIVE_KEYWORDS)
    if neg > pos: return "negative"
    elif pos > neg: return "positive"
    return "neutral"

df_valid['sentimen_label'] = df_valid['teks_ternormalisasi'].apply(label_sentiment)
print("Distribusi Sentimen:")
print(df_valid['sentimen_label'].value_counts())

# Ambil ulasan negatif (keluhan)
neg_mask = df_valid['sentimen_label'] == 'negative'
neg_indices = np.where(neg_mask)[0]
neg_tfidf = tfidf_matrix[neg_indices]
print(f"\nJumlah ulasan keluhan yang di-cluster: {neg_tfidf.shape[0]}")

# Reduksi Dimensi dengan TruncatedSVD
svd = TruncatedSVD(n_components=10, random_state=42)
reduced_neg_tfidf = svd.fit_transform(neg_tfidf)
print(f"Explained Variance TruncatedSVD: {svd.explained_variance_ratio_.sum():.2%}")

# Clustering K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
complaint_clusters = kmeans.fit_predict(reduced_neg_tfidf)
df_valid.loc[neg_mask, 'complaint_cluster'] = complaint_clusters

# Ekstraksi Top Kata Kunci Keluhan per Klaster
cluster_centers = svd.inverse_transform(kmeans.cluster_centers_)
for i, center in enumerate(cluster_centers):
    top_kw_idx = center.argsort()[::-1][:7]
    top_words = [feature_names[idx] for idx in top_kw_idx]
    print(f"\nKelompok Keluhan {i}: {', '.join(top_words)}")

Distribusi Sentimen:
sentimen_label
neutral     917
positive    276
negative    109
Name: count, dtype: int64

Jumlah ulasan keluhan yang di-cluster: 109
Explained Variance TruncatedSVD: 19.87%

Kelompok Keluhan 0: panas, buang, suhu, layar, keyboardnya, buang panas, acer

Kelompok Keluhan 1: rusak, mati, loq, layar, kalau, turun, laptop

Kelompok Keluhan 2: kurang, laptop, banget, kalau, sama, sesal, game


## 8. Penyimpanan Dataset & Model Artefak

Menyimpan dataset hasil preprocessing lengkap dan model (TF-IDF Vectorizer, TruncatedSVD, KMeans) agar dapat langsung diintegrasikan pada model prediksi sentimen dan API backend.

In [10]:
# Simpan dataset bersih
out_path = "../raw-dataset/dataset_preprocessed_review.csv"
if not os.path.exists("../raw-dataset"):
    out_path = "raw-dataset/dataset_preprocessed_review.csv"

df_valid.to_csv(out_path, index=False)
print(f"Dataset preprocessing berhasil disimpan di: {out_path}")

# Simpan model vektorizer dan clustering
model_dir = "../model" if os.path.exists("../model") else "model"
joblib.dump(tfidf_vectorizer, os.path.join(model_dir, "tfidf_review_vectorizer.pkl"))
joblib.dump(svd, os.path.join(model_dir, "svd_clustering.pkl"))
joblib.dump(kmeans, os.path.join(model_dir, "kmeans_complaints.pkl"))
print(f"Model berhasil disimpan ke direktori: {model_dir}/")

Dataset preprocessing berhasil disimpan di: ../raw-dataset/dataset_preprocessed_review.csv
Model berhasil disimpan ke direktori: ../model/


---
## 9. Simpan Hasil Preprocessing ke Database

Masukkan teks ulasan dan label sentimen ke tabel `reviews` dan `sentiment_results`. ID komentar YouTube disimpan agar proses dapat dijalankan ulang tanpa membuat duplikat.

In [11]:
import sqlite3

required_columns = {'ID_Komentar', 'Sumber', 'Teks_Asli', 'teks_final', 'sentimen_label'}
missing_columns = required_columns - set(df_valid.columns)
if missing_columns:
    raise ValueError(f"Kolom hasil preprocessing tidak tersedia: {sorted(missing_columns)}")

df_reviews_db = pd.DataFrame({
    'id_sumber': df_valid['ID_Komentar'].astype(str),
    'sumber': df_valid['Sumber'],
    'teks_asli': df_valid['Teks_Asli'],
    'teks_bersih': df_valid['teks_final'],
})
if df_reviews_db['id_sumber'].duplicated().any():
    raise ValueError('ID_Komentar duplikat ditemukan pada hasil preprocessing.')

# Dukung eksekusi dari folder root proyek maupun folder notebooks.
database_path = os.path.join('..', 'data', 'gamergear.db')
if not os.path.isfile(database_path):
    database_path = os.path.join('data', 'gamergear.db')

with sqlite3.connect(database_path) as conn:
    review_columns = {row[1] for row in conn.execute("PRAGMA table_info('reviews')")}
    if not review_columns:
        raise RuntimeError(f"Tabel 'reviews' tidak ditemukan di {database_path}. Jalankan init_db.py terlebih dahulu.")
    if 'id_sumber' not in review_columns:
        conn.execute('ALTER TABLE reviews ADD COLUMN id_sumber TEXT')
    conn.execute('''
        CREATE UNIQUE INDEX IF NOT EXISTS idx_reviews_source_id
        ON reviews (sumber, id_sumber)
    ''')

    existing_ids = set(conn.execute(
        'SELECT sumber, id_sumber FROM reviews WHERE id_sumber IS NOT NULL'
    ).fetchall())
    rows_to_insert = []
    for row in df_reviews_db.itertuples(index=False):
        record_key = (row.sumber, row.id_sumber)
        if record_key not in existing_ids:
            rows_to_insert.append(row)
            existing_ids.add(record_key)

    if rows_to_insert:
        pd.DataFrame(rows_to_insert, columns=df_reviews_db.columns).to_sql(
            'reviews', conn, if_exists='append', index=False
        )

    review_ids = {
        (source, source_id): review_id
        for review_id, source, source_id in conn.execute(
            'SELECT id, sumber, id_sumber FROM reviews WHERE id_sumber IS NOT NULL'
        )
    }
    existing_sentiments = set(conn.execute(
        "SELECT review_id, aspek FROM sentiment_results WHERE aspek = 'umum'"
    ).fetchall())
    sentiment_rows = []
    for row in df_valid[['ID_Komentar', 'Sumber', 'sentimen_label']].itertuples(index=False):
        review_id = review_ids[(row.Sumber, str(row.ID_Komentar))]
        if (review_id, 'umum') not in existing_sentiments:
            sentiment_rows.append((review_id, 'umum', row.sentimen_label, None))
            existing_sentiments.add((review_id, 'umum'))

    if sentiment_rows:
        pd.DataFrame(
            sentiment_rows,
            columns=['review_id', 'aspek', 'label_sentimen', 'skor']
        ).to_sql('sentiment_results', conn, if_exists='append', index=False)

print(f"✅ {len(rows_to_insert)} ulasan ditambahkan ke tabel reviews.")
print(f"✅ {len(sentiment_rows)} label sentimen ditambahkan ke tabel sentiment_results.")
print(f"Data yang sudah ada dan dilewati: {len(df_reviews_db) - len(rows_to_insert)} ulasan.")

✅ 0 ulasan ditambahkan ke tabel reviews.
✅ 0 label sentimen ditambahkan ke tabel sentiment_results.
Data yang sudah ada dan dilewati: 1302 ulasan.
